# PC Build Advisor using Open Pre-trained Transformers (OPT)

Revised from the supplied notebook. This prototype extracts English, Malay and Manglish PC requirements, uses deterministic budget/compatibility rules to recommend PC components and optional peripherals, and generates explanations with an OPT LoRA adapter.

**Run instructions:** Put `csv.zip` next to the notebook (or upload when requested in Colab). Use a T4 GPU for `full` mode, which retains the three-model comparison. `quick` mode is a smaller OPT smoke experiment; `baseline` mode runs only rule-based components, and is not the full OPT experiment. Use Runtime > Run all. UI launch is optional; model weights require Hugging Face access. If Colab already imported a conflicting Transformers version, restart the runtime before running again.

**Limitations:** USD prices and fixed USD/MYR rate of 4.3 are assumptions from the original notebook, not verified current Malaysian prices. Compatibility covers encoded rules only, not all physical clearances, BIOS revisions, cooler mounting or actual benchmark performance. Synthetic training/test labels and the small OOD set do not establish real-world accuracy. Lexical explanation grounding is not a complete factuality check. Review and declare AI assistance in accordance with the assignment instructions. No full OPT results are prefilled in this revised notebook.

## 1. Setup and execution mode

In [ ]:
# Configure before running all cells. Google Colab: Runtime > Change runtime type > T4 GPU.
import importlib.util, sys, subprocess, os
import torch
RUN_MODE = "full" if torch.cuda.is_available() else "baseline"  # full / quick / baseline
LAUNCH_UI = False  # Set True to open Gradio; keep False for unattended Run all.
SHARE_UI = False   # Set True only if a public Colab share link is needed.
assert RUN_MODE in {"full", "quick", "baseline"}
if RUN_MODE != "baseline":
    # Pin compatible versions instead of upgrading to an incompatible latest release.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformers==4.51.3", "peft==0.15.2", "accelerate==1.6.0"])
    import transformers
    print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if RUN_MODE == "baseline":
    print("Baseline mode: OPT training/evaluation is skipped. GPU full mode is needed for the complete assignment experiment.")
SIZES = ["125m", "350m", "1.3b"] if RUN_MODE == "full" else (["125m"] if RUN_MODE == "quick" else [])
EXPLAIN_SIZE = SIZES[-1] if SIZES else None


## 2. Data cleaning and compatibility engine

In [ ]:
%%writefile pc_engine.py
"""
pc_engine.py
============
Data cleaning + knowledge-based PC build engine.

The CSV files only contain raw part listings. Several things that a real
build needs are NOT in the data, so this module derives them with explicit,
documented rules (these are the "knowledge base" of the system):

  * CPU socket          -> derived from `microarchitecture` (+ name)
  * Motherboard chipset -> parsed from the board name
  * Memory type (DDR)   -> parsed from `speed` ("5,6000" = DDR5 @ 6000 MT/s);
                           for boards: from the socket, or from the board name
  * GPU performance/TDP -> small curated table (GPU_TABLE, approximate)
  * Case <-> board fit  -> mapping of case type to supported board sizes
  * PSU wattage         -> (CPU TDP + GPU TDP + 100 W) * 1.25 headroom

All prices in the CSVs are USD; they are converted to MYR for display.
"""
import re
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd

USD_TO_MYR = 4.3          # change if the exchange rate changes
SUPPORTED_SOCKETS = {"AM4", "AM5", "LGA1200", "LGA1700", "LGA1851"}

# --------------------------------------------------------------------------
# Curated GPU table: chipset -> (relative performance, board power in watts)
# Relative performance is approximate (RTX 4090 = 100), compiled from public
# review aggregates. It is an ASSUMPTION of the system, not part of the CSV.
# --------------------------------------------------------------------------
GPU_TABLE = {
    # NVIDIA RTX 50
    "GeForce RTX 5090": (130, 575), "GeForce RTX 5080": (85, 360),
    "GeForce RTX 5070 Ti": (78, 300), "GeForce RTX 5070": (58, 250),
    "GeForce RTX 5060 Ti": (45, 180), "GeForce RTX 5060": (38, 145),
    "GeForce RTX 5050": (28, 130),
    # NVIDIA RTX 40
    "GeForce RTX 4090": (100, 450), "GeForce RTX 4080 SUPER": (78, 320),
    "GeForce RTX 4080": (76, 320), "GeForce RTX 4070 Ti SUPER": (67, 285),
    "GeForce RTX 4070 Ti": (61, 285), "GeForce RTX 4070 SUPER": (58, 220),
    "GeForce RTX 4070": (50, 200), "GeForce RTX 4060 Ti": (40, 160),
    "GeForce RTX 4060": (33, 115),
    # NVIDIA RTX 30
    "GeForce RTX 3090 Ti": (67, 450), "GeForce RTX 3090": (62, 350),
    "GeForce RTX 3080 Ti": (61, 350), "GeForce RTX 3080 12GB LHR": (59, 350),
    "GeForce RTX 3080 10GB": (55, 320), "GeForce RTX 3080 10GB LHR": (55, 320),
    "GeForce RTX 3070 Ti": (46, 290), "GeForce RTX 3070": (43, 220),
    "GeForce RTX 3070 LHR": (43, 220), "GeForce RTX 3060 Ti": (38, 200),
    "GeForce RTX 3060 Ti LHR": (38, 200), "GeForce RTX 3060 12GB": (28, 170),
    "GeForce RTX 3060 8GB": (26, 170), "GeForce RTX 3050 8GB": (20, 130),
    "GeForce RTX 3050 6GB": (17, 70),
    # NVIDIA RTX 20 / GTX 16 / GTX 10
    "GeForce RTX 2080 Ti": (42, 250), "GeForce RTX 2080 SUPER": (37, 250),
    "GeForce RTX 2080": (35, 215), "GeForce RTX 2070 SUPER": (33, 215),
    "GeForce RTX 2070": (29, 175), "GeForce RTX 2060 SUPER": (27, 175),
    "GeForce RTX 2060": (23, 160), "GeForce GTX 1660 Ti": (20, 120),
    "GeForce GTX 1660 SUPER": (20, 125), "GeForce GTX 1660": (18, 120),
    "GeForce GTX 1650 SUPER": (17, 100), "GeForce GTX 1650 G6": (12, 75),
    "GeForce GTX 1080 Ti": (33, 250), "GeForce GTX 1080": (25, 180),
    "GeForce GTX 1070 Ti": (22, 180), "GeForce GTX 1070": (20, 150),
    "GeForce GTX 1060 6GB": (13.5, 120), "GeForce GTX 1050 Ti": (9, 75),
    # AMD RX 9000 / 7000 / 6000 / 5000
    "Radeon RX 9070 XT": (70, 304), "Radeon RX 9070": (63, 220),
    "Radeon RX 9060 XT": (40, 160), "Radeon RX 7900 XTX": (70, 355),
    "Radeon RX 7900 XT": (61, 315), "Radeon RX 7900 GRE": (55, 260),
    "Radeon RX 7800 XT": (52, 263), "Radeon RX 7700 XT": (45, 245),
    "Radeon RX 7600 XT": (32, 190), "Radeon RX 7600": (29, 165),
    "Radeon RX 6950 XT": (55, 335), "Radeon RX 6900 XT": (52, 300),
    "Radeon RX 6800 XT": (50, 300), "Radeon RX 6800": (45, 250),
    "Radeon RX 6750 XT": (38, 250), "Radeon RX 6700 XT": (36, 230),
    "Radeon RX 6650 XT": (27, 180), "Radeon RX 6600 XT": (26, 160),
    "Radeon RX 6600": (24, 132), "Radeon RX 6500 XT": (14, 107),
    "Radeon RX 6400": (11, 53), "Radeon RX 5700 XT": (31, 225),
    "Radeon RX 5600 XT": (24, 150), "Radeon RX 5500 XT": (15, 130),
    "Radeon RX 580": (13, 185), "Radeon RX 570": (11, 150),
    # Intel Arc
    "Arc B580": (34, 190), "Arc B570": (30, 150), "Arc A770": (30, 225),
    "Arc A750": (27, 225), "Arc A580": (23, 185),
}

# CPU microarchitecture -> (socket, relative IPC/generation factor)
ARCH_INFO = {
    "Zen": ("AM4", 0.65), "Zen+": ("AM4", 0.72), "Zen 2": ("AM4", 0.85),
    "Zen 3": ("AM4", 1.00), "Zen 4": ("AM5", 1.15), "Zen 5": ("AM5", 1.28),
    "Comet Lake": ("LGA1200", 0.85), "Rocket Lake": ("LGA1200", 0.95),
    "Alder Lake": ("LGA1700", 1.05), "Raptor Lake": ("LGA1700", 1.15),
    "Raptor Lake Refresh": ("LGA1700", 1.18), "Arrow Lake": ("LGA1851", 1.20),
}

CHIPSET_RE = re.compile(
    r"(?<![A-Z0-9])(A520|B450|B550|X570|A620|B650|X670|B840|B850|X870|"
    r"H410|B460|H470|Z490|H510|B560|H570|Z590|H610|B660|H670|Z690|B760|"
    r"H770|Z790|H810|B860|Z890)(?![0-9])")
CHIPSET_TIER = {  # 1 = entry, 2 = mainstream, 3 = enthusiast
    "A520": 1, "B450": 1, "A620": 1, "H410": 1, "H510": 1, "H610": 1,
    "H810": 1, "B840": 1, "B550": 2, "B650": 2, "B460": 2, "B560": 2,
    "B660": 2, "B760": 2, "B860": 2, "H470": 2, "H570": 2, "H670": 2,
    "H770": 2, "B850": 3, "X570": 3, "X670": 3, "X870": 3, "Z490": 3,
    "Z590": 3, "Z690": 3, "Z790": 3, "Z890": 3,
}

# Which case types can hold which motherboard form factors (towers only,
# so that a discrete GPU physically fits).
_ATX_CASES = {"ATX Mid Tower", "ATX Full Tower", "ATX Mini Tower"}
_MATX_CASES = _ATX_CASES | {"MicroATX Mini Tower", "MicroATX Mid Tower"}
_ITX_CASES = _MATX_CASES | {"Mini ITX Tower"}
CASE_SUPPORT = {"ATX": _ATX_CASES, "Micro ATX": _MATX_CASES,
                "Mini ITX": _ITX_CASES}
COMPACT_CASES = {"MicroATX Mini Tower", "MicroATX Mid Tower", "Mini ITX Tower"}

EFF_RANK = {"plus": 1, "bronze": 2, "silver": 3, "gold": 4,
            "platinum": 5, "titanium": 6}

# CPU usefulness stops growing beyond this normalised score (light workloads)
CPU_SATURATION = dict(office=0.35, general=0.50, programming=0.80)

MIN_CORES = dict(gaming=6, streaming=8, video_editing=8, ai_ml=6,
                 programming=6, office=4, general=4)

USE_CASES = ["gaming", "video_editing", "office", "programming",
             "streaming", "ai_ml", "general"]

# use case -> (cpu weight, gpu weight, cpu score kind, needs discrete GPU,
#              default ram GB, default storage GB)
PROFILES = {
    "gaming":        dict(wc=0.35, wg=0.65, cpu="gaming", dgpu=True,  ram=16, ssd=1000),
    "video_editing": dict(wc=0.50, wg=0.50, cpu="multi",  dgpu=True,  ram=32, ssd=1000),
    "streaming":     dict(wc=0.50, wg=0.50, cpu="multi",  dgpu=True,  ram=32, ssd=1000),
    "ai_ml":         dict(wc=0.30, wg=0.70, cpu="multi",  dgpu=True,  ram=32, ssd=1000),
    "programming":   dict(wc=0.90, wg=0.10, cpu="multi",  dgpu=False, ram=32, ssd=1000),
    "office":        dict(wc=1.00, wg=0.00, cpu="multi",  dgpu=False, ram=16, ssd=500),
    "general":       dict(wc=0.85, wg=0.15, cpu="multi",  dgpu=False, ram=16, ssd=500),
}

NUM = lambda x: pd.to_numeric(x, errors="coerce")


# --------------------------------------------------------------------------
# 1. Loading and cleaning
# --------------------------------------------------------------------------
def _read(csv_dir, name):
    return pd.read_csv(os.path.join(csv_dir, name + ".csv"))


def _priced_unique(df):
    """Keep only rows with a price and the cheapest row per product name."""
    df = df.copy()
    df["price"] = NUM(df["price"])
    df = df[df["price"] > 0].dropna(subset=["price", "name"])
    return df.sort_values("price").drop_duplicates("name", keep="first")


def clean_cpu(df):
    df = _priced_unique(df)
    n = df["name"].str.lower()
    df = df[~n.str.contains("xeon|threadripper|epyc|pro \\d")]
    info = df["microarchitecture"].map(ARCH_INFO)
    df = df[info.notna()].copy()
    info = info[df.index]
    df["socket"] = [i[0] for i in info]
    df["arch_factor"] = [i[1] for i in info]
    df["brand"] = np.where(df["name"].str.contains("AMD"), "amd", "intel")
    df["boost"] = NUM(df["boost_clock"]).fillna(NUM(df["core_clock"]))
    df["cores"] = NUM(df["core_count"])
    df["tdp"] = NUM(df["tdp"])
    df["has_igpu"] = df["graphics"].notna()
    df["x3d"] = df["name"].str.contains("X3D")
    # hybrid Intel (12th gen+): E-cores count roughly half
    hybrid = df["microarchitecture"].isin(
        ["Alder Lake", "Raptor Lake", "Raptor Lake Refresh", "Arrow Lake"])
    df["eff_cores"] = np.where(hybrid & (df["cores"] > 8),
                               8 + (df["cores"] - 8) * 0.5, df["cores"])
    df["gaming_raw"] = (df["boost"] * df["arch_factor"]
                        * np.where(df["x3d"], 1.25, 1.0)
                        * np.minimum(df["cores"], 8) ** 0.3)
    df["multi_raw"] = df["eff_cores"] ** 0.9 * df["boost"] * df["arch_factor"]
    df["gaming_score"] = df["gaming_raw"] / df["gaming_raw"].max()
    df["multi_score"] = df["multi_raw"] / df["multi_raw"].max()
    # coolers: unlocked / X-series chips normally ship WITHOUT a cooler
    df["needs_cooler"] = (df["name"].str.contains(r"(?:K|KF|KS|X|X3D|XT)$")
                          | (df["tdp"] >= 105))
    return df.reset_index(drop=True)


def clean_gpu(df):
    df = _priced_unique(df)
    df = df[df["chipset"].isin(GPU_TABLE)].copy()
    df["perf"] = df["chipset"].map(lambda c: GPU_TABLE[c][0])
    df["tdp"] = df["chipset"].map(lambda c: GPU_TABLE[c][1])
    df["vram"] = NUM(df["memory"])
    df["brand"] = np.where(df["chipset"].str.startswith("GeForce"), "nvidia",
                   np.where(df["chipset"].str.startswith("Radeon"), "amd",
                            "intel"))
    df = df[df["vram"].notna()].copy()
    # cheapest card per (chipset, VRAM size) keeps the search space small
    df = df.sort_values("price").drop_duplicates(["chipset", "vram"], keep="first")
    # more VRAM than the base 8 GB gives a small bonus (and less a penalty)
    adj = 1 + 0.08 * np.log2(np.maximum(df["vram"], 4) / 8.0)
    df["score"] = df["perf"] * adj / 130.0          # RTX 5090 ~ 1.0
    return df.reset_index(drop=True)


def _board_ddr(row):
    """Return (ddr_generation, confirmed). None = cannot be inferred."""
    s, name, chip = row["socket"], row["name"], row["chipset_code"]
    if s in ("AM4", "LGA1200"):
        return 4, True
    if s in ("AM5", "LGA1851"):
        return 5, True
    if s == "LGA1700":   # supports both generations -> look at the name
        if re.search(r"DDR\s?4|\bD4\b|[-/ ]D4", name, re.I):
            return 4, True
        if re.search(r"DDR\s?5|\bD5\b", name, re.I):
            return 5, True
        if chip == "H610":
            return 4, False       # assumed
        if chip in ("Z790", "Z690"):
            return 5, False       # assumed
    return None, False


def clean_motherboard(df):
    df = _priced_unique(df)
    df = df[df["socket"].isin(SUPPORTED_SOCKETS)].copy()
    df["chipset_code"] = df["name"].str.upper().str.extract(CHIPSET_RE)[0]
    df = df[df["chipset_code"].notna()]
    df = df[df["form_factor"].isin(["ATX", "Micro ATX", "Mini ITX"])].copy()
    ddr = df.apply(_board_ddr, axis=1)
    df["ddr"] = [d[0] for d in ddr]
    df["ddr_confirmed"] = [d[1] for d in ddr]
    df = df[df["ddr"].notna()].copy()
    df["tier"] = df["chipset_code"].map(CHIPSET_TIER)
    df["max_mem"] = NUM(df["max_memory"])
    df["slots"] = NUM(df["memory_slots"])
    return df.reset_index(drop=True)


def clean_memory(df):
    df = _priced_unique(df)
    sp = df["speed"].astype(str).str.split(",", expand=True)
    mo = df["modules"].astype(str).str.split(",", expand=True)
    df["ddr"] = NUM(sp[0])
    df["mhz"] = NUM(sp[1])
    df["n_mod"] = NUM(mo[0])
    df["gb_each"] = NUM(mo[1])
    df["total_gb"] = df["n_mod"] * df["gb_each"]
    df = df[((df["ddr"] == 4) & (df["mhz"] >= 3200)) |
            ((df["ddr"] == 5) & (df["mhz"] >= 5600))]
    df = df[df["n_mod"].isin([1, 2])]
    df = df[(df["n_mod"] == 2) | (df["total_gb"] <= 8)]
    return df.reset_index(drop=True)


def clean_storage(df):
    df = _priced_unique(df)
    df["capacity"] = NUM(df["capacity"])
    df = df[(df["type"] == "SSD") & (df["form_factor"] == "M.2-2280")
            & df["interface"].astype(str).str.startswith("M.2 PCIe")]
    df = df[df["capacity"].between(250, 4000)]
    return df.reset_index(drop=True)


PSU_BRANDS = ("Corsair", "SeaSonic", "Seasonic", "EVGA", "be quiet!", "Thermaltake",
              "MSI", "Cooler Master", "Gigabyte", "Asus", "NZXT", "Antec", "FSP",
              "Super Flower", "Fractal Design", "Lian Li", "SilverStone",
              "Silverstone", "XPG", "DeepCool", "Phanteks", "ADATA")


def clean_psu(df):
    df = _priced_unique(df)
    df = df[df["name"].str.startswith(PSU_BRANDS)]   # reputable brands only
    df["watt"] = NUM(df["wattage"])
    df["eff"] = df["efficiency"].map(EFF_RANK)
    df = df[(df["type"] == "ATX") & (df["eff"] >= 2) & (df["watt"] >= 350)]
    return df.reset_index(drop=True)


def clean_case(df):
    df = _priced_unique(df)
    df = df[df["type"].isin(_ITX_CASES) & (df["price"] >= 30)]
    return df.reset_index(drop=True)


def clean_cooler(df):
    df = _priced_unique(df)
    df = df[df["size"].isna() & df["noise_level"].notna() & (df["price"] >= 15)]
    return df.reset_index(drop=True)


def clean_monitor(df):
    df = _priced_unique(df)
    res = df["resolution"].astype(str).str.split(",", expand=True)
    df["w"], df["h"] = NUM(res[0]), NUM(res[1])
    df["size_in"] = NUM(df["screen_size"])
    df["hz"] = NUM(df["refresh_rate"])
    df = df[df["h"].notna() & df["hz"].notna() & df["size_in"].between(21, 34)
            & (df["h"] >= 1080)]
    return df.reset_index(drop=True)


def clean_keyboard(df):
    df = _priced_unique(df)
    return df[df["price"] >= 12].reset_index(drop=True)


def clean_mouse(df):
    df = _priced_unique(df)
    df["dpi"] = NUM(df["max_dpi"])
    return df[df["price"] >= 8].reset_index(drop=True)


def load_data(csv_dir):
    """Load every CSV the engine needs and return cleaned DataFrames."""
    d = dict(
        cpu=clean_cpu(_read(csv_dir, "cpu")),
        gpu=clean_gpu(_read(csv_dir, "video-card")),
        mobo=clean_motherboard(_read(csv_dir, "motherboard")),
        ram=clean_memory(_read(csv_dir, "memory")),
        ssd=clean_storage(_read(csv_dir, "internal-hard-drive")),
        psu=clean_psu(_read(csv_dir, "power-supply")),
        case=clean_case(_read(csv_dir, "case")),
        cooler=clean_cooler(_read(csv_dir, "cpu-cooler")),
        monitor=clean_monitor(_read(csv_dir, "monitor")),
        keyboard=clean_keyboard(_read(csv_dir, "keyboard")),
        mouse=clean_mouse(_read(csv_dir, "mouse")),
    )
    return d


def cleaning_report(csv_dir, data):
    """Rows before/after cleaning for the report."""
    files = dict(cpu="cpu", gpu="video-card", mobo="motherboard",
                 ram="memory", ssd="internal-hard-drive", psu="power-supply",
                 case="case", cooler="cpu-cooler", monitor="monitor",
                 keyboard="keyboard", mouse="mouse")
    rows = []
    for k, f in files.items():
        raw = _read(csv_dir, f)
        rows.append(dict(category=k, raw_rows=len(raw),
                         missing_price_pct=round(100 * raw["price"].isna().mean(), 1),
                         usable_rows=len(data[k])))
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# 2. Compatibility knowledge
# --------------------------------------------------------------------------
def chipset_supports_cpu(chipset, cpu_arch, socket):
    """Out-of-the-box BIOS support rules (simplified)."""
    if socket == "AM4":
        if cpu_arch in ("Zen 3",):
            return chipset in ("B550", "X570", "A520")
        return chipset in ("B550", "X570", "A520", "B450")
    if socket == "LGA1200":
        if cpu_arch == "Rocket Lake":
            return chipset in ("B560", "H510", "H570", "Z590")
        return True
    return True


def psu_required_watts(cpu_tdp, gpu_tdp):
    raw = (cpu_tdp + gpu_tdp + 100) * 1.25
    return int(np.ceil(raw / 50.0) * 50)


# --------------------------------------------------------------------------
# 3. Build engine
# --------------------------------------------------------------------------
@dataclass
class Requirement:
    use_case: str = "general"
    budget: float = 4000.0
    currency: str = "MYR"
    cpu_brand: str = "any"
    gpu_brand: str = "any"
    ram_gb: int = None
    storage_gb: int = None
    extras: tuple = ()
    compact: bool = False

    @classmethod
    def from_dict(cls, d):
        uc = d.get("use_case") or "general"
        if uc not in USE_CASES:
            uc = "general"
        ex = d.get("extras") or []
        if isinstance(ex, str):
            ex = [ex]
        return cls(
            use_case=uc,
            budget=float(d.get("budget") or 4000),
            currency=(d.get("currency") or "MYR").upper(),
            cpu_brand=(d.get("cpu_brand") or "any").lower(),
            gpu_brand=(d.get("gpu_brand") or "any").lower(),
            ram_gb=int(d["ram_gb"]) if d.get("ram_gb") else None,
            storage_gb=int(d["storage_gb"]) if d.get("storage_gb") else None,
            extras=tuple(e for e in ex if e in ("monitor", "keyboard", "mouse")),
            compact=bool(d.get("compact")),
        )

    @property
    def budget_usd(self):
        return self.budget if self.currency == "USD" else self.budget / USD_TO_MYR


class BuildEngine:
    def __init__(self, data):
        self.d = data
        self._ram_cache = {}
        self._plat_cache = {}
        self._psu_cache = {}

    # ---- helpers ---------------------------------------------------------
    def _cheapest_ram(self, ddr, gb):
        key = (ddr, gb)
        if key not in self._ram_cache:
            r = self.d["ram"]
            r = r[(r["ddr"] == ddr) & (r["total_gb"] == gb)]
            self._ram_cache[key] = None if len(r) == 0 else \
                r.sort_values("price").iloc[0]
        return self._ram_cache[key]

    def _platform(self, cpu, req, ram_gb, min_tier):
        key = (cpu["socket"], cpu["microarchitecture"], req.compact, ram_gb, min_tier)
        if key not in self._plat_cache:
            self._plat_cache[key] = self._platform_uncached(cpu, req, ram_gb, min_tier)
        return self._plat_cache[key]

    def _platform_uncached(self, cpu, req, ram_gb, min_tier):
        """Cheapest (board, ram) pair compatible with the CPU, or None."""
        b = self.d["mobo"]
        b = b[(b["socket"] == cpu["socket"]) & (b["tier"] >= min_tier)]
        b = b.loc[b["chipset_code"].map(lambda c: chipset_supports_cpu(
            c, cpu["microarchitecture"], cpu["socket"])).astype(bool)]
        if req.compact:
            b = b[b["form_factor"].isin(["Mini ITX", "Micro ATX"])]
        b = b[(b["max_mem"] >= ram_gb) & (b["slots"] >= 2)]
        best = None
        for ddr in b["ddr"].unique():
            ram = self._cheapest_ram(int(ddr), ram_gb)
            if ram is None:
                continue
            board = b[b["ddr"] == ddr].sort_values("price").iloc[0]
            tot = board["price"] + ram["price"]
            if best is None or tot < best[0]:
                best = (tot, board, ram)
        return best

    def _case(self, ff, compact):
        c = self.d["case"]
        c = c[c["type"].isin(CASE_SUPPORT[ff])]
        if compact:
            c = c[c["type"].isin(COMPACT_CASES)]
        return None if len(c) == 0 else c.sort_values("price").iloc[0]

    def _ssd(self, gb):
        s = self.d["ssd"]
        s = s[(s["capacity"] >= gb) & (s["capacity"] <= gb * 2.1)]
        return None if len(s) == 0 else s.sort_values("price").iloc[0]

    def _psu(self, watts, min_eff):
        key = (watts, min_eff)
        if key not in self._psu_cache:
            p = self.d["psu"]
            p = p[(p["watt"] >= watts) & (p["eff"] >= min_eff)]
            self._psu_cache[key] = None if len(p) == 0 else \
                p.sort_values("price").iloc[0]
        return self._psu_cache[key]

    def _cooler(self, cpu):
        if not cpu["needs_cooler"]:
            return None
        c = self.d["cooler"]
        floor = 40 if cpu["tdp"] >= 125 else 20
        c = c[c["price"] >= floor]
        return None if len(c) == 0 else c.sort_values("price").iloc[0]

    # ---- extras ----------------------------------------------------------
    def _pick_extras(self, req, budget_usd):
        caps = dict(monitor=0.15, keyboard=0.03, mouse=0.02)
        out = {}
        for e in req.extras:
            cap = budget_usd * caps[e]
            if e == "keyboard":
                cap = min(cap, 80)
            elif e == "mouse":
                cap = min(cap, 60)
            if e == "monitor":
                m = self.d["monitor"]
                m = m[m["price"] <= cap]
                if req.use_case == "office" or req.use_case == "general":
                    m = m[m["hz"] >= 60]
                    score = (m["w"] * m["h"] / 2.0e6) + 0.2 * (m["panel_type"] == "IPS")
                elif req.use_case in ("gaming", "streaming"):
                    m = m[m["hz"] >= 100]
                    score = (m["w"] * m["h"] / 2.0e6) * 0.5 + m["hz"] / 144 \
                        + 0.2 * (m["panel_type"] == "IPS")
                else:  # editing / programming / ai
                    score = (m["w"] * m["h"] / 2.0e6) + 0.5 * (m["panel_type"] == "IPS")
                if len(m):
                    out["monitor"] = m.loc[score.idxmax()]
            elif e == "keyboard":
                k = self.d["keyboard"]
                k = k[(k["price"] <= cap) & ~k["name"].str.contains(
                    "combo|kit|set|bundle", case=False)]
                if req.use_case in ("gaming", "streaming") and \
                        (k["style"] == "Gaming").any():
                    k = k[k["style"] == "Gaming"]
                if len(k):
                    i = (k["price"] - 0.7 * cap).abs().idxmin()
                    out["keyboard"] = k.loc[i]
            elif e == "mouse":
                m = self.d["mouse"]
                m = m[(m["price"] <= cap) & m["dpi"].notna()]
                if req.use_case in ("gaming", "streaming") and (m["dpi"] >= 8000).any():
                    m = m[m["dpi"] >= 8000]
                if len(m):
                    i = (m["price"] - 0.7 * cap).abs().idxmin()
                    out["mouse"] = m.loc[i]
        return out

    # ---- main search -----------------------------------------------------
    def recommend(self, req):
        if isinstance(req, dict):
            req = Requirement.from_dict(req)
        prof = PROFILES[req.use_case]
        total_usd = req.budget_usd

        extras = self._pick_extras(req, total_usd)
        extras_cost = sum(float(r["price"]) for r in extras.values())
        core_budget = total_usd - extras_cost

        ram_gb = req.ram_gb or prof["ram"]
        if ram_gb > 64:
            return dict(ok=False, requirement=req, message="This prototype supports up to 64 GB RAM.")
        ssd_gb = req.storage_gb or prof["ssd"]

        # small budgets: scale the defaults down instead of failing
        if not req.ram_gb and core_budget < 700:
            ram_gb = 16 if core_budget >= 450 else 8
        if not req.storage_gb and core_budget < 700:
            ssd_gb = 500

        ssd = self._ssd(ssd_gb) if ssd_gb else None
        if ssd is None:
            return dict(ok=False, requirement=req, message="No SSD meets the requested capacity in this catalogue.")

        cpus = self.d["cpu"]
        if req.cpu_brand in ("amd", "intel"):
            cpus = cpus[cpus["brand"] == req.cpu_brand]

        gpus = self.d["gpu"]
        if req.gpu_brand in ("nvidia", "amd", "intel"):
            gpus = gpus[gpus["brand"] == req.gpu_brand]
        if req.use_case == "ai_ml":
            gpus = gpus[gpus["vram"] >= 12] if (gpus["vram"] >= 12).any() else gpus

        # light-use profiles run on integrated graphics unless a GPU brand
        # was explicitly requested
        wants_gpu = prof["dgpu"] or req.gpu_brand != "any"
        if wants_gpu:
            gpu_options = [g for _, g in gpus.iterrows()]
        else:
            gpu_options = [None]
            cpus = cpus[cpus["has_igpu"]]
        cpu_key = "gaming_score" if prof["cpu"] == "gaming" else "multi_score"

        strict = cpus[cpus["cores"] >= MIN_CORES[req.use_case]]
        best, cheapest = self._search(strict, gpu_options, req, prof, cpu_key,
                                      ram_gb, ssd, core_budget)
        if best is None:                      # budget too small -> relax
            best, cheapest = self._search(cpus, gpu_options, req, prof, cpu_key,
                                          ram_gb, ssd, core_budget)

        if best is None:
            mn = None if cheapest is None else (cheapest[-2] + extras_cost) * USD_TO_MYR
            return dict(ok=False, requirement=req, min_cost_myr=mn,
                        message=("Budget is too low for a compatible build"
                                 + ("" if mn is None else
                                    f"; the cheapest build in the dataset is about RM {mn:,.0f}.")))

        _, _, cpu, gpu, board, ram, case, cooler, psu, total, watts = best
        parts = dict(cpu=cpu, mobo=board, ram=ram, ssd=ssd, case=case, psu=psu)
        if gpu is not None:
            parts["gpu"] = gpu
        if cooler is not None:
            parts["cooler"] = cooler
        parts.update(extras)

        # spend what is left on a better PSU / more RAM / more storage
        left = core_budget - total
        parts, left, upgrades = self._upgrade(parts, left, req, ram_gb, ssd_gb, watts)

        return self._pack(req, parts, watts, left, upgrades)

    def _search(self, cpus, gpu_options, req, prof, cpu_key, ram_gb, ssd,
                core_budget):
        best, cheapest = None, None
        for _, cpu in cpus.iterrows():
            min_tier = 2 if (cpu["tdp"] >= 105 or cpu["cores"] >= 12) else 1
            plat = self._platform(cpu, req, ram_gb, min_tier)
            if plat is None:
                continue
            _, board, ram = plat
            case = self._case(board["form_factor"], req.compact)
            if case is None:
                continue
            cooler = self._cooler(cpu)
            if cpu["needs_cooler"] and cooler is None:
                continue
            fixed = (cpu["price"] + board["price"] + ram["price"]
                     + case["price"] + (ssd["price"] if ssd is not None else 0)
                     + (cooler["price"] if cooler is not None else 0))
            for gpu in gpu_options:
                if gpu is None and not cpu["has_igpu"]:
                    continue
                gtdp = 0 if gpu is None else gpu["tdp"]
                watts = psu_required_watts(cpu["tdp"], gtdp)
                psu = self._psu(watts, 4 if gtdp >= 250 else 2)
                if psu is None:
                    continue
                total = fixed + psu["price"] + (0 if gpu is None else gpu["price"])
                obj = self._objective(req.use_case, prof, cpu, gpu, cpu_key)
                obj *= (1 - 0.12 * total / max(core_budget, 1))   # mild thrift
                cand = (obj, -total, cpu, gpu, board, ram, case, cooler, psu,
                        total, watts)
                if cheapest is None or total < cheapest[-2]:
                    cheapest = cand
                if total <= core_budget and (best is None or cand[:2] > best[:2]):
                    best = cand

        return best, cheapest

    @staticmethod
    def _objective(use_case, prof, cpu, gpu, cpu_key):
        """Estimated usefulness of a CPU+GPU pair for the use case."""
        c = min(float(cpu[cpu_key]), CPU_SATURATION.get(use_case, 9.0))
        if gpu is None:                      # integrated graphics build
            return c
        g = float(gpu["score"])
        if prof["cpu"] == "gaming":
            # FPS is limited by the slower of CPU and GPU -> soft minimum
            c, p = c * 1.3, 3.0
            return (c ** -p + g ** -p) ** (-1.0 / p)
        if use_case == "ai_ml":
            g *= 0.5 + 0.5 * min(float(gpu["vram"]), 24) / 24
            if gpu["brand"] == "nvidia":
                g *= 1.25          # CUDA ecosystem
        wg = max(prof["wg"], 0.3)                         # user asked for a GPU
        return (c ** (1 - wg)) * (g ** wg)                # weighted geometric mean

    def _upgrade(self, parts, left, req, ram_gb, ssd_gb, watts):
        notes = []
        # 1) PSU -> gold if affordable
        cur = parts["psu"]
        if cur["eff"] < 4:
            alt = self._psu(watts, 4)
            if alt is not None and alt["price"] - cur["price"] <= left * 0.5:
                left -= alt["price"] - cur["price"]
                parts["psu"] = alt
                notes.append("PSU upgraded to 80+ Gold with leftover budget")
        # 2) storage -> next size up
        for nxt in (500, 1000, 2000, 4000):
            if nxt <= ssd_gb:
                continue
            alt = self._ssd(nxt)
            if alt is not None and alt["price"] - parts["ssd"]["price"] <= left * 0.6:
                left -= alt["price"] - parts["ssd"]["price"]
                parts["ssd"] = alt
                notes.append(f"SSD increased to {int(alt['capacity'])} GB")
            break
        # 3) RAM -> 32 GB if on 16
        if ram_gb == 16 and not req.ram_gb:
            ram = self._cheapest_ram(int(parts["ram"]["ddr"]), 32)
            if ram is not None and ram["price"] - parts["ram"]["price"] <= left * 0.8 \
                    and parts["mobo"]["max_mem"] >= 32:
                left -= ram["price"] - parts["ram"]["price"]
                parts["ram"] = ram
                notes.append("RAM increased to 32 GB")
        # 4) tempered-glass case for gaming/streaming builds
        if req.use_case in ("gaming", "streaming"):
            ff = parts["mobo"]["form_factor"]
            c = self.d["case"]
            c = c[c["type"].isin(CASE_SUPPORT[ff]) & (c["side_panel"] == "Tempered Glass")]
            if req.compact:
                c = c[c["type"].isin(COMPACT_CASES)]
            if len(c):
                alt = c.sort_values("price").iloc[0]
                if 0 < alt["price"] - parts["case"]["price"] <= left * 0.6:
                    left -= alt["price"] - parts["case"]["price"]
                    parts["case"] = alt
                    notes.append("Case changed to a tempered-glass model")
        return parts, left, notes

    # ---- output ------------------------------------------------------------
    def _pack(self, req, parts, watts, left, upgrades):
        order = ["cpu", "cooler", "mobo", "ram", "gpu", "ssd", "psu", "case",
                 "monitor", "keyboard", "mouse"]
        items, total = [], 0.0
        for k in order:
            if k not in parts:
                continue
            r = parts[k]
            total += float(r["price"])
            items.append(dict(category=k, name=r["name"],
                              price_usd=float(r["price"]),
                              price_myr=float(r["price"]) * USD_TO_MYR,
                              spec=self._spec(k, r)))
        warnings = []
        if left / max(req.budget_usd, 1) > 0.35 and req.use_case in ("office", "general", "programming"):
            warnings.append("This use case does not need the whole budget, so the "
                            "build stops early. Name a GPU brand or a higher use "
                            "case (e.g. gaming) if you want to spend more.")
        missing = sorted(set(req.extras) - set(parts))
        if missing:
            warnings.append("Requested peripherals not included within their allocated budget: " + ", ".join(missing))
        b = parts["mobo"]
        if not b["ddr_confirmed"]:
            warnings.append("Memory type of the motherboard was inferred from its "
                            "chipset; verify DDR4/DDR5 on the manufacturer page.")
        warnings.append("CPU-cooler socket support and GPU length vs case "
                        "clearance are not in the dataset - verify before buying.")
        return dict(ok=True, requirement=req, items=items,
                    total_usd=total, total_myr=total * USD_TO_MYR,
                    budget_myr=req.budget_usd * USD_TO_MYR,
                    left_myr=left * USD_TO_MYR, psu_min_watts=watts,
                    upgrades=upgrades, warnings=warnings)

    @staticmethod
    def _spec(k, r):
        f = lambda v: "" if pd.isna(v) else v
        if k == "cpu":
            return (f"{int(r['cores'])} cores, up to {r['boost']:.1f} GHz, "
                    f"{int(r['tdp'])} W, socket {r['socket']}"
                    + (", integrated graphics" if r["has_igpu"] else ""))
        if k == "gpu":
            return f"{r['chipset']}, {int(r['vram'])} GB VRAM, ~{int(r['tdp'])} W"
        if k == "mobo":
            return f"{r['chipset_code']} chipset, {r['socket']}, {r['form_factor']}, DDR{int(r['ddr'])}"
        if k == "ram":
            return f"{int(r['total_gb'])} GB ({int(r['n_mod'])}x{int(r['gb_each'])}) DDR{int(r['ddr'])}-{int(r['mhz'])}"
        if k == "ssd":
            return f"{int(r['capacity'])} GB NVMe, {r['interface']}"
        if k == "psu":
            return f"{int(r['watt'])} W, 80+ {r['efficiency']}, {f(r['modular'])} modular"
        if k == "case":
            return f"{r['type']}, {f(r['side_panel'])}"
        if k == "cooler":
            return f"air cooler, {f(r['noise_level'])} dBA"
        if k == "monitor":
            return f"{r['size_in']:.0f}\" {int(r['w'])}x{int(r['h'])} {int(r['hz'])} Hz {f(r['panel_type'])}"
        if k == "keyboard":
            return f"{f(r['style'])} keyboard, {f(r['connection_type'])}"
        if k == "mouse":
            return f"mouse, up to {int(r['dpi']) if not pd.isna(r['dpi']) else '?'} DPI"
        return ""


# --------------------------------------------------------------------------
# 4. Presentation helpers
# --------------------------------------------------------------------------
CAT_LABEL = dict(cpu="CPU", cooler="CPU cooler", mobo="Motherboard", ram="Memory",
                 gpu="Graphics card", ssd="Storage", psu="Power supply",
                 case="Case", monitor="Monitor", keyboard="Keyboard", mouse="Mouse")


def format_build(res):
    if not res["ok"]:
        return res["message"]
    lines = []
    for it in res["items"]:
        lines.append(f"{CAT_LABEL[it['category']]:<14} {it['name']}  "
                     f"[{it['spec']}]  RM {it['price_myr']:,.0f}")
    lines.append("-" * 70)
    lines.append(f"TOTAL RM {res['total_myr']:,.0f}  (budget RM {res['budget_myr']:,.0f}, "
                 f"left RM {res['left_myr']:,.0f})")
    for u in res["upgrades"]:
        lines.append("* " + u)
    for w in res["warnings"]:
        lines.append("! " + w)
    return "\n".join(lines)


def facts_text(res):
    """Compact fact sheet fed to OPT for grounded explanation generation."""
    r = res["requirement"]
    lines = [f"Use: {r.use_case.replace('_', ' ')} | Budget: RM {res['budget_myr']:,.0f} | "
             f"Total: RM {res['total_myr']:,.0f}"]
    for it in res["items"]:
        lines.append(f"{CAT_LABEL[it['category']]}: {it['name']} | {it['spec']} | "
                     f"RM {it['price_myr']:,.0f}")
    lines.append(f"Estimated power requirement: {res['psu_min_watts']} W")
    return "\n".join(lines)


_REASON = {
    "gaming": "Most of the budget goes to the graphics card because frame rate in games depends on it the most.",
    "video_editing": "Video editing needs many CPU cores and plenty of RAM, so the budget is split between a strong CPU and GPU, with as much RAM as the budget allows.",
    "streaming": "Streaming needs strong multi-core CPU power to encode while the GPU renders the game.",
    "ai_ml": "AI work depends on GPU memory (VRAM), so a graphics card with as much VRAM as the budget allows was chosen.",
    "programming": "Programming and compiling benefit from CPU cores and RAM, while a powerful GPU is not needed.",
    "office": "Office work only needs a modest CPU with integrated graphics, which keeps the cost low.",
    "general": "A balanced CPU with enough RAM and fast SSD storage suits everyday use.",
}


def template_explanation(res):
    """Rule-based explanation. Used as training target AND as safe fallback."""
    if not res["ok"]:
        return res["message"]
    r = res["requirement"]
    g = {it["category"]: it for it in res["items"]}
    s = [_REASON[r.use_case]]
    c = g["cpu"]
    s.append(f"The {c['name']} ({c['spec']}) was chosen as the processor.")
    if "gpu" in g:
        s.append(f"The {g['gpu']['name']} ({g['gpu']['spec']}) handles graphics.")
    else:
        s.append("No separate graphics card is needed because the CPU has integrated graphics.")
    s.append(f"The {g['mobo']['name']} is compatible with the CPU socket and uses "
             f"{g['ram']['spec'].split(')')[-1].strip().split('-')[0]} memory.")
    s.append(f"Storage is a {g['ssd']['spec'].split(',')[0]} and the "
             f"{g['psu']['name']} provides {g['psu']['spec'].split(',')[0]}, "
             f"above the estimated {res['psu_min_watts']} W requirement.")
    s.append(f"The total is RM {res['total_myr']:,.0f} against a budget of "
             f"RM {res['budget_myr']:,.0f}.")
    return " ".join(s)


# --------------------------------------------------------------------------
# 5. Independent validator + randomised compatibility test
# --------------------------------------------------------------------------
def validate_build(res, data):
    """Re-check a finished build against every rule. Returns violation list."""
    if not res["ok"]:
        return []
    req = res["requirement"]
    names = {it["category"]: it for it in res["items"]}
    look = lambda key, cat: data[key][data[key]["name"] == names[cat]["name"]].iloc[0]
    cpu, mb, ram = look("cpu", "cpu"), look("mobo", "mobo"), look("ram", "ram")
    psu, case = look("psu", "psu"), look("case", "case")
    v = []
    if cpu["socket"] != mb["socket"]:
        v.append("socket mismatch")
    if not chipset_supports_cpu(mb["chipset_code"], cpu["microarchitecture"], cpu["socket"]):
        v.append("chipset does not support CPU generation")
    if int(ram["ddr"]) != int(mb["ddr"]):
        v.append("DDR generation mismatch")
    if ram["total_gb"] > mb["max_mem"]:
        v.append("RAM exceeds board maximum")
    if ram["n_mod"] > mb["slots"]:
        v.append("more RAM modules than slots")
    if case["type"] not in CASE_SUPPORT[mb["form_factor"]]:
        v.append("board does not fit case")
    if req.compact and case["type"] not in COMPACT_CASES:
        v.append("compact requested but case is not compact")
    if "gpu" in names:
        gpu = look("gpu", "gpu")
        gtdp = gpu["tdp"]
        if req.gpu_brand in ("nvidia", "amd", "intel") and gpu["brand"] != req.gpu_brand:
            v.append("GPU brand preference ignored")
    else:
        gtdp = 0
        if not cpu["has_igpu"]:
            v.append("no GPU and CPU has no integrated graphics")
    if psu["watt"] < psu_required_watts(cpu["tdp"], gtdp):
        v.append("PSU too weak")
    if req.cpu_brand in ("amd", "intel") and cpu["brand"] != req.cpu_brand:
        v.append("CPU brand preference ignored")
    if res["total_myr"] > res["budget_myr"] + 0.01:
        v.append("over budget")
    if req.ram_gb and int(ram["total_gb"]) < req.ram_gb:
        v.append("less RAM than requested")
    if "ssd" not in names:
        v.append("missing SSD")
    elif req.storage_gb and look("ssd", "ssd")["capacity"] < req.storage_gb:
        v.append("less storage than requested")
    if cpu["needs_cooler"] and "cooler" not in names:
        v.append("missing required CPU cooler")
    # Optional peripherals may be omitted; _pack reports these explicitly.
    return v


def run_compat_tests(engine, n=300, seed=0):
    """Random requirements -> build -> validate. Returns summary DataFrame."""
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(n):
        req = dict(
            use_case=str(rng.choice(USE_CASES)),
            budget=float(rng.choice(np.arange(1500, 15001, 500))),
            currency="MYR",
            cpu_brand=str(rng.choice(["any", "any", "amd", "intel"])),
            gpu_brand=str(rng.choice(["any", "any", "nvidia", "amd", "intel"])),
            ram_gb=int(rng.choice([0, 0, 16, 32, 64])) or None,
            storage_gb=int(rng.choice([0, 0, 500, 1000, 2000])) or None,
            extras=[e for e in ("monitor", "keyboard", "mouse") if rng.random() < 0.25],
            compact=bool(rng.random() < 0.15),
        )
        res = engine.recommend(req)
        viol = validate_build(res, engine.d)
        rows.append(dict(**{k: (str(v) if isinstance(v, list) else v)
                            for k, v in req.items()},
                         built=res["ok"], violations=len(viol),
                         violation_text="; ".join(viol)))
    return pd.DataFrame(rows)

## 3. Multilingual NLU dataset, prompts and baseline

In [ ]:
%%writefile nlu_data.py
"""
nlu_data.py
===========
Natural-language understanding side of the PC Build Advisor.

* SCHEMA / canonical JSON format that OPT must produce
* Synthetic English / Malay / Manglish ("rojak") request generator
  - phrases starting with '*' are HELD OUT: they never appear in the
    training split and appear in ~50% of the test split, so the test
    set measures generalisation, not memorisation.
* Rule-based (regex / keyword) baseline extractor
* Prompt builders and evaluation metrics
"""
import json
import re
import random

KEYS = ["use_case", "budget", "currency", "cpu_brand", "gpu_brand",
        "ram_gb", "storage_gb", "extras", "compact"]

# ---------------------------------------------------------------- phrase banks
USE = {
    "gaming": [
        ("gaming", "en"), ("playing AAA games", "en"), ("esports", "en"),
        ("playing Valorant and CS2", "en"), ("*playing Cyberpunk and Elden Ring", "en"),
        ("main game", "ms"), ("game berat", "ms"), ("game online", "ms"),
        ("*main game FPS", "ms"), ("gaming lah", "mix"), ("play games one", "mix"),
        ("*game Dota", "mix")],
    "video_editing": [
        ("video editing", "en"), ("editing videos", "en"), ("3D rendering", "en"),
        ("content creation", "en"), ("*Premiere Pro and After Effects", "en"),
        ("edit video", "ms"), ("edit video YouTube", "ms"), ("*render video", "ms"),
        ("edit video lah", "mix"), ("do content creation", "mix"),
        ("*Davinci Resolve", "mix")],
    "office": [
        ("office work", "en"), ("Excel and Word", "en"), ("school assignments", "en"),
        ("*spreadsheets and emails", "en"), ("kerja pejabat", "ms"),
        ("buat assignment", "ms"), ("kerja office", "ms"), ("*kerja Excel", "ms"),
        ("office work only", "mix"), ("study one", "mix"), ("*do assignment", "mix")],
    "programming": [
        ("programming", "en"), ("coding", "en"), ("software development", "en"),
        ("*web development and Docker", "en"), ("buat coding", "ms"),
        ("develop aplikasi", "ms"), ("*program Python", "ms"),
        ("coding lah", "mix"), ("do programming", "mix"), ("*code Java", "mix")],
    "streaming": [
        ("streaming", "en"), ("live streaming on Twitch", "en"),
        ("game streaming", "en"), ("*streaming on YouTube", "en"),
        ("buat live stream", "ms"), ("streaming game", "ms"),
        ("*live stream TikTok", "ms"), ("stream game lah", "mix"),
        ("*stream Twitch", "mix")],
    "ai_ml": [
        ("machine learning", "en"), ("AI model training", "en"),
        ("deep learning", "en"), ("running local LLMs", "en"),
        ("*data science with PyTorch", "en"), ("belajar machine learning", "ms"),
        ("latih model AI", "ms"), ("*kerja deep learning", "ms"),
        ("do AI stuff", "mix"), ("run LLM", "mix"), ("*train neural network", "mix")],
    "general": [
        ("daily use", "en"), ("general use", "en"),
        ("browsing and watching Netflix", "en"), ("everyday tasks", "en"),
        ("*light everyday work", "en"), ("guna harian", "ms"), ("guna biasa", "ms"),
        ("tengok Netflix", "ms"), ("*guna rumah", "ms"), ("normal use only", "mix"),
        ("surf internet", "mix"), ("*basic use", "mix")],
}
OPEN = {
    "en": ["I want to build a PC for {u}.", "Can you recommend PC parts for {u}?",
           "Need a computer for {u}.", "Help me pick components for a {u} PC.",
           "*Looking to put together a desktop for {u}."],
    "ms": ["Saya nak build PC untuk {u}.", "Tolong cadangkan komponen komputer untuk {u}.",
           "Nak rakit PC untuk {u}.", "Boleh cadangkan PC untuk {u}?",
           "*Saya perlukan komputer baru untuk {u}."],
    "mix": ["Boss, I want build PC for {u} lah.", "Bro can recommend PC for {u} or not?",
            "Eh got any PC build for {u} ah?", "Wah need new PC for {u} la.",
            "*Hi ah, planning to build one PC for {u} leh."],
}
BUDGET = {
    ("MYR", "en"): ["My budget is RM{n}", "Budget RM{n}", "around RM{n}",
                    "under RM{n}", "*not more than RM{n}"],
    ("MYR", "ms"): ["bajet RM{n}", "bawah RM{n}", "dalam RM{n}",
                    "lebih kurang RM{n}", "*maksimum RM{n}", "bajet {n} ringgit"],
    ("MYR", "mix"): ["budget RM{n} only", "RM{n} can ah", "{n} ringgit la",
                     "budget {n} only", "*budget around RM{n} la"],
    ("USD", "en"): ["my budget is ${n}", "budget ${n}", "under ${n}",
                    "{n} USD", "*around {n} dollars"],
}
CPU = {"amd": ["AMD CPU", "Ryzen processor", "guna AMD", "*Ryzen only"],
       "intel": ["Intel CPU", "Intel Core processor", "guna Intel", "*Intel chip please"]}
GPU = {"nvidia": ["Nvidia GPU", "RTX card", "kad grafik Nvidia", "*NVIDIA graphics"],
       "amd": ["Radeon GPU", "AMD graphics card", "kad grafik Radeon", "*AMD GPU"],
       "intel": ["Intel Arc GPU", "Arc graphics"]}
BOTH = {("amd", "amd"): ["all AMD build", "semua AMD", "full AMD", "*AMD everything"],
        ("intel", "nvidia"): ["Intel and Nvidia combo", "Intel CPU with Nvidia card",
                              "*Intel + RTX"]}
RAM = ["{n}GB RAM", "RAM {n}GB", "{n} GB of memory", "ram {n}g", "*{n}gig memory"]
SSD = {500: ["500GB SSD", "storage 500GB", "SSD 500GB"],
       1000: ["1TB SSD", "storage 1TB", "SSD 1000GB", "*1 TB NVMe"],
       2000: ["2TB SSD", "2TB storage", "SSD 2000GB", "*2 TB NVMe"]}
EXTRA_WORDS = {"monitor": ["monitor", "monitor", "*display"],
               "keyboard": ["keyboard", "keyboard", "papan kekunci"],
               "mouse": ["mouse", "mouse", "tetikus"]}
EXTRA_JOIN = {"en": ["with {x}", "include {x}", "also need {x}"],
              "ms": ["dengan {x}", "nak {x} sekali"],
              "mix": ["{x} sekali", "{x} also can", "got {x} or not"]}
COMPACT = ["small form factor", "compact build", "mini ITX", "PC kecil",
           "small casing", "*as small as possible"]


def _pick(rng, items, mode, key=lambda x: x):
    """Train mode never sees '*' items; test mode sees them ~50% of the time."""
    star = [i for i in items if key(i).startswith("*")]
    normal = [i for i in items if not key(i).startswith("*")]
    if mode == "test" and star and rng.random() < 0.5:
        choice = rng.choice(star)
    else:
        choice = rng.choice(normal)
    return choice


def _clean(s):
    return s[1:] if s.startswith("*") else s


# ------------------------------------------------------------------ generator
def make_sample(rng, mode="train"):
    """Return (text, label_dict)."""
    lang = rng.choice(["en", "ms", "mix"], ) if False else rng.choice(["en", "ms", "mix"])
    uc = rng.choice(list(USE))
    uphrase, ulang = _pick(rng, USE[uc], mode, key=lambda t: t[0])
    uphrase = _clean(uphrase)
    opener = _clean(_pick(rng, OPEN[lang], mode)).format(u=uphrase)

    cur = "USD" if rng.random() < 0.15 else "MYR"
    if cur == "USD":
        n = rng.choice(range(500, 3501, 100))
        bl = "en"
    else:
        n = rng.choice(list(range(1500, 15001, 100)))
        bl = lang
    shown = str(n)
    r = rng.random()
    if n % 1000 == 0 and r < 0.35:
        shown = f"{n // 1000}k"
    elif n >= 1000 and r < 0.55:
        shown = f"{n:,}"
    elif cur == "MYR" and n % 500 == 0 and n >= 2000 and r < 0.75:
        shown = f"{n / 1000:g}k"
    if cur == "MYR" and bl != "en" and n % 1000 == 0 and rng.random() < 0.25:
        bphrase = rng.choice(["bajet {k} ribu", "dalam {k} ribu", "lebih kurang RM{k} ribu"]
                             ).format(k=n // 1000)
    else:
        bphrase = _clean(_pick(rng, BUDGET[(cur, bl)], mode)).format(n=shown)

    label = dict(use_case=uc, budget=n, currency=cur, cpu_brand="any",
                 gpu_brand="any", ram_gb=None, storage_gb=None, extras=[],
                 compact=False)
    clauses = [bphrase]

    if rng.random() < 0.40:
        t = rng.random()
        if t < 0.25:
            pair = rng.choice(list(BOTH))
            label["cpu_brand"], label["gpu_brand"] = pair
            clauses.append(_clean(_pick(rng, BOTH[pair], mode)))
        else:
            if rng.random() < 0.5:
                b = rng.choice(list(CPU))
                label["cpu_brand"] = b
                clauses.append(_clean(_pick(rng, CPU[b], mode)))
            if rng.random() < 0.7 or label["cpu_brand"] == "any":
                b = rng.choice(list(GPU))
                label["gpu_brand"] = b
                clauses.append(_clean(_pick(rng, GPU[b], mode)))
    if rng.random() < 0.35:
        g = rng.choice([8, 16, 32, 64])
        label["ram_gb"] = g
        clauses.append(_clean(_pick(rng, RAM, mode)).format(n=g))
    if rng.random() < 0.35:
        g = rng.choice([500, 1000, 2000])
        label["storage_gb"] = g
        clauses.append(_clean(_pick(rng, SSD[g], mode)))
    if rng.random() < 0.22:
        ex = [e for e in EXTRA_WORDS if rng.random() < 0.5] or [rng.choice(list(EXTRA_WORDS))]
        words = [_clean(_pick(rng, EXTRA_WORDS[e], mode)) for e in ex]
        label["extras"] = [e for e in EXTRA_WORDS if e in ex]
        x = ", ".join(words[:-1]) + (" and " if len(words) > 1 else "") + words[-1]
        clauses.append(_clean(rng.choice(EXTRA_JOIN[lang])).format(x=x))
    if rng.random() < 0.08:
        label["compact"] = True
        clauses.append(_clean(_pick(rng, COMPACT, mode)))

    rng.shuffle(clauses)
    tail = ""
    for i, c in enumerate(clauses):
        tail += ("" if i == 0 else rng.choice([", ", ". ", " and ", "; "])) + c
    text = opener + " " + tail.strip() + rng.choice([".", "", "!", " thanks."])
    if rng.random() < 0.12:
        text = text.lower()
    return text, label


def build_dataset(n, seed=0, mode="train"):
    rng = random.Random(seed)
    out, seen = [], set()
    while len(out) < n:
        t, l = make_sample(rng, mode)
        if t in seen:
            continue
        seen.add(t)
        out.append((t, l))
    return out


# ------------------------------------------------------- JSON / prompt format
def to_json(label):
    """Canonical compact JSON with fixed key order."""
    return json.dumps({k: label[k] for k in KEYS}, separators=(",", ":"))


def normalise(d):
    """Make a parsed dict comparable to the gold label (fill + coerce)."""
    out = dict(use_case="general", budget=None, currency="MYR", cpu_brand="any",
               gpu_brand="any", ram_gb=None, storage_gb=None, extras=[],
               compact=False)
    if not isinstance(d, dict):
        return out
    for k in KEYS:
        if k in d:
            out[k] = d[k]
    try:
        out["budget"] = int(float(out["budget"])) if out["budget"] is not None else None
    except (TypeError, ValueError, OverflowError):
        out["budget"] = None
    for k in ("ram_gb", "storage_gb"):
        try:
            out[k] = int(float(out[k])) if out[k] is not None else None
        except (TypeError, ValueError, OverflowError):
            out[k] = None
    if not isinstance(out["extras"], list):
        out["extras"] = []
    out["extras"] = sorted(set(str(e) for e in out["extras"]) & {"monitor", "keyboard", "mouse"})
    for key, choices, default in [
        ("use_case", set(USE), "general"), ("currency", {"MYR", "USD"}, "MYR"),
        ("cpu_brand", {"any", "intel", "amd"}, "any"),
        ("gpu_brand", {"any", "nvidia", "amd", "intel"}, "any")]:
        val = out[key]
        out[key] = val if isinstance(val, str) and val in choices else default
    for key in ("budget", "ram_gb", "storage_gb"):
        if out[key] is not None and out[key] <= 0:
            out[key] = None
    out["compact"] = out["compact"] is True or str(out["compact"]).lower() == "true"
    return out


def parse_model_output(text):
    """Extract the first JSON object from generated text. None if invalid."""
    text = text.strip()
    start = text.find("{")
    if start < 0:
        return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None


EXTRACT_HEAD = ("Extract the PC requirements from the customer message as JSON.\n"
                "Keys: use_case, budget, currency, cpu_brand, gpu_brand, ram_gb, "
                "storage_gb, extras, compact.\n")


def extract_prompt(text):
    return f"{EXTRACT_HEAD}Message: {text}\nJSON:"


def few_shot_prompt(text, shots):
    body = "".join(f"Message: {t}\nJSON: {to_json(l)}\n\n" for t, l in shots)
    return f"{EXTRACT_HEAD}\n{body}Message: {text}\nJSON:"


def explain_prompt(facts):
    return f"Recommended PC build:\n{facts}\nExplain why this build suits the customer:"


# ----------------------------------------------------------- rule baseline
_UC_RULES = [
    ("ai_ml", r"machine learning|deep learning|\bai\b|\bllm|neural|pytorch|data science|latih model"),
    ("streaming", r"stream|twitch|live ?stream"),
    ("video_editing", r"edit(ing)? video|video edit|render|premiere|davinci|content creation|after effects"),
    ("programming", r"coding|\bcode\b|programming|\bprogram\b|develop|docker"),
    ("gaming", r"gaming|\bgame|esports|valorant|cs2|dota|fps|cyberpunk"),
    ("office", r"office|excel|word|assignment|spreadsheet|pejabat|study"),
    ("general", r"daily|general|netflix|everyday|harian|biasa|browsing|surf|basic|rumah"),
]


def rule_extract(text):
    """Keyword / regex baseline (no learning)."""
    t = text.lower()
    out = dict(use_case="general", budget=None, currency="MYR", cpu_brand="any",
               gpu_brand="any", ram_gb=None, storage_gb=None, extras=[], compact=False)
    for uc, pat in _UC_RULES:
        if re.search(pat, t):
            out["use_case"] = uc
            break
    m = re.search(r"(?:rm|\$)\s?([\d][\d,\.]*)\s?(k|ribu)?(?![a-z])|([\d][\d,\.]*)\s?(k|ribu)?(?![a-z])\s?(?:ringgit|usd|dollars|myr)", t)
    if not m:
        m = re.search(r"(?:budget|bajet)\s?(?:is |around |only )?([\d][\d,\.]*)\s?(k|ribu)?(?![a-z])", t)
    if m:
        groups = [g for g in m.groups() if g]
        num = groups[0].replace(",", "")
        try:
            val = float(num.rstrip(".,")) * (1000 if len(groups) > 1 and groups[-1] in ("k", "ribu") else 1)
            out["budget"] = int(val)
        except ValueError:
            pass
    if "$" in t or re.search(r"\busd\b|dollars", t):
        out["currency"] = "USD"
    if re.search(r"all amd|semua amd|full amd|amd everything", t):
        out["cpu_brand"] = out["gpu_brand"] = "amd"
    elif re.search(r"intel and nvidia|intel cpu with nvidia|intel \+ rtx", t):
        out["cpu_brand"], out["gpu_brand"] = "intel", "nvidia"
    else:
        if re.search(r"ryzen|amd cpu|guna amd", t):
            out["cpu_brand"] = "amd"
        elif re.search(r"intel (cpu|core|chip)|guna intel", t):
            out["cpu_brand"] = "intel"
        if re.search(r"nvidia|rtx|geforce", t):
            out["gpu_brand"] = "nvidia"
        elif re.search(r"radeon|amd graphics|amd gpu", t):
            out["gpu_brand"] = "amd"
        elif re.search(r"arc", t):
            out["gpu_brand"] = "intel"
    m = re.search(r"(\d+)\s?(?:gb|g)\s?(?:of )?(?:ram|memory)|ram\s?(\d+)\s?gb?|(\d+)\s?gig", t)
    if m:
        out["ram_gb"] = int([g for g in m.groups() if g][0])
    m = re.search(r"(\d+(?:\.\d+)?)\s?(tb|gb)\s?(?:ssd|storage|nvme)|(?:ssd|storage)\s?(\d+(?:\.\d+)?)\s?(tb|gb)", t)
    if m:
        g = [x for x in m.groups() if x]
        v = float(g[0]) * (1000 if g[1] == "tb" else 1)
        out["storage_gb"] = int(v)
    for e, pat in (("monitor", r"monitor|display"), ("keyboard", r"keyboard|papan kekunci"),
                   ("mouse", r"mouse|tetikus")):
        if re.search(pat, t):
            out["extras"].append(e)
    out["compact"] = bool(re.search(r"small form|compact|mini itx|pc kecil|small casing|as small", t))
    return out


# ------------------------------------------------------------------ metrics
def evaluate(preds, golds):
    """preds: list of dict|None (None = invalid JSON). golds: label dicts."""
    n = len(golds)
    if n == 0 or len(preds) != n:
        raise ValueError("Predictions and gold labels must have equal nonzero length")
    valid = sum(p is not None for p in preds)
    field_ok = {k: 0 for k in KEYS}
    exact = 0
    for p, g in zip(preds, golds):
        pn, gn = normalise(p), normalise(g)
        ok_all = p is not None
        for k in KEYS:
            ok = (pn[k] == gn[k]) and p is not None
            field_ok[k] += ok
            ok_all &= ok
        exact += ok_all
    res = {"json_valid_%": 100 * valid / n, "exact_match_%": 100 * exact / n}
    for k in KEYS:
        res[f"{k}_%"] = 100 * field_ok[k] / n
    res["mean_field_acc_%"] = sum(100 * field_ok[k] / n for k in KEYS) / len(KEYS)
    return res


# ------------------------------------------------------------------ OOD set
def _L(uc, budget, cur="MYR", cpu="any", gpu="any", ram=None, ssd=None,
       extras=(), compact=False):
    return dict(use_case=uc, budget=budget, currency=cur, cpu_brand=cpu,
                gpu_brand=gpu, ram_gb=ram, storage_gb=ssd,
                extras=sorted(extras), compact=compact)


# Hand-written, natural-sounding requests that do NOT come from the generator
# templates. Used to measure out-of-distribution robustness.
OOD_SET = [
    ("my son wants a pc to play fortnite and minecraft, we can spend about rm3500", _L("gaming", 3500)),
    ("Looking for a workstation for Blender and Unreal Engine, 12k ringgit max, nvidia please", _L("video_editing", 12000, gpu="nvidia")),
    ("recommend me a build for my office staff, excel and chrome only, rm 2000", _L("office", 2000)),
    ("I'd like to train small neural nets at home, budget 1800 usd, prefer nvidia", _L("ai_ml", 1800, "USD", gpu="nvidia")),
    ("boss, rm5000 got any good gaming pc? intel cpu and 32gb ram", _L("gaming", 5000, cpu="intel", ram=32)),
    ("pc untuk edit video tiktok, bajet 4 ribu", _L("video_editing", 4000)),
    ("nak buat live stream dan main game, budget rm 6,500, nak monitor sekali", _L("streaming", 6500, extras=["monitor"])),
    ("Can I get a decent PC for web dev around $1,200? AMD CPU please", _L("programming", 1200, "USD", cpu="amd")),
    ("budget RM 3k, just for browsing youtube and zoom meetings", _L("general", 3000)),
    ("Need something small, mini itx, for gaming. rm4500.", _L("gaming", 4500, compact=True)),
    ("pls recommend pc 2tb ssd 64gb ram for data science, rm 14000", _L("ai_ml", 14000, ram=64, ssd=2000)),
    ("I have 8k to spend. want to edit 4k videos with keyboard and mouse included.", _L("video_editing", 8000, extras=["keyboard", "mouse"])),
    ("sis nak PC untuk kerja akaun, tak payah mahal, rm1800", _L("office", 1800)),
    ("gaming pc rm 2.5k, amd gpu", _L("gaming", 2500, gpu="amd")),
    ("Radeon only and ryzen, for playing Call of Duty, RM 6000", _L("gaming", 6000, cpu="amd", gpu="amd")),
    ("Trying to spec a PC to run Llama locally, 3000 USD, 64GB of RAM", _L("ai_ml", 3000, "USD", ram=64)),
    ("student pc for assignments and Zoom classes, RM 2200", _L("office", 2200)),
    ("wah my budget only 1.5k ah, can build gaming pc or not", _L("gaming", 1500)),
    ("twitch streamer here, need a build, ten thousand ringgit, 1tb ssd", _L("streaming", 10000, ssd=1000)),
    ("Recommend a PC for coding and docker containers, 16GB RAM enough, RM4000", _L("programming", 4000, ram=16)),
    ("build for editing youtube vids, budget about 5000 myr, nvidia gpu", _L("video_editing", 5000, gpu="nvidia")),
    ("cheap pc for my mom to watch netflix and facebook, below rm 1800", _L("general", 1800)),
    ("I want a high-end gaming rig, money is no object, about RM 20000", _L("gaming", 20000)),
    ("Budget: 4,500 ringgit. Use: coding. Intel CPU + RTX 4060", _L("programming", 4500, cpu="intel", gpu="nvidia")),
    ("pc game sekitar rm4k, ram 16gb, ssd 1tb", _L("gaming", 4000, ram=16, ssd=1000)),
    ("need pc for rendering in 3ds max and v-ray, budget USD 2200", _L("video_editing", 2200, "USD")),
    ("just a basic desktop for the shop counter, rm1600", _L("office", 1600)),
    ("I stream on Twitch daily, 9000 MYR, with monitor keyboard and mouse", _L("streaming", 9000, extras=["monitor", "keyboard", "mouse"])),
    ("Intel Arc GPU build for esports, 3000 ringgit", _L("gaming", 3000, gpu="intel")),
    ("Fine-tune LLMs at home, RM 15000, Nvidia only", _L("ai_ml", 15000, gpu="nvidia")),
    ("Can you suggest a quiet PC for general use? budget $900", _L("general", 900, "USD")),
    ("mak nak PC untuk tengok drama dan video call, bajet rm2k", _L("general", 2000)),
    ("program in C++ and run virtual machines, 32GB RAM, rm 5500", _L("programming", 5500, ram=32)),
    ("1TB SSD for office use, rm2300", _L("office", 2300, ssd=1000)),
    ("Gaming PC under 3k with Nvidia", _L("gaming", 3000, gpu="nvidia")),
    ("I'm into esports (Valorant), need a small PC, compact, budget RM 3500, ryzen", _L("gaming", 3500, cpu="amd", compact=True)),
    ("Computer for gaming and I also need a monitor, RM 6500", _L("gaming", 6500, extras=["monitor"])),
]

## 4. OPT inference and LoRA training utilities

In [ ]:
%%writefile opt_nlu.py
"""
opt_nlu.py
==========
Everything that touches Meta's OPT models (Hugging Face port).

  load_opt()            load a base OPT model in fp16
  generate()            batched greedy generation
  run_extraction()      zero-shot / few-shot requirement extraction
  finetune_lora()       LoRA fine-tuning with prompt-masked loss
  load_finetuned()      base + adapter -> merged model for inference
  OptNLU                inference wrapper (extract + explain + fallbacks)
  check_grounded()      lexical hallucination check for explanations
"""
import gc
import json
import re
import time

import torch

import nlu_data as N

OPT_SIZES = {"125m": "facebook/opt-125m", "350m": "facebook/opt-350m",
             "1.3b": "facebook/opt-1.3b", "2.7b": "facebook/opt-2.7b"}


# ------------------------------------------------------------------ loading
def load_opt(name, dtype=None):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    dtype = dtype or (torch.float16 if torch.cuda.is_available() else torch.float32)
    tok = AutoTokenizer.from_pretrained(name, use_fast=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    return tok, model


def free():
    """Call AFTER `del model, tok` in the caller (del inside a function does
    not drop the caller's references)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def n_params(model):
    return sum(p.numel() for p in model.parameters())


# --------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, tok, prompts, max_new_tokens=100, batch_size=8, stop=None):
    """Greedy decoding. Returns only the newly generated text per prompt."""
    outs = []
    tok.padding_side = "left"
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i + batch_size]
        context = getattr(model.config, "max_position_embeddings", 2048)
        tok.truncation_side = "left"
        enc = tok(chunk, return_tensors="pt", padding=True, truncation=True,
                  max_length=context - max_new_tokens).to(model.device)
        kw = dict(max_new_tokens=max_new_tokens, do_sample=False,
                  pad_token_id=tok.pad_token_id)
        if stop:
            kw.update(stop_strings=stop, tokenizer=tok)
        try:
            seq = model.generate(**enc, **kw)
        except (TypeError, ValueError):       # old transformers: no stop_strings
            kw.pop("stop_strings", None)
            kw.pop("tokenizer", None)
            seq = model.generate(**enc, **kw)
        new = seq[:, enc["input_ids"].shape[1]:]
        outs += [tok.decode(s, skip_special_tokens=True) for s in new]
    return outs


def pick_shots(train, k=4, seed=0):
    """k training examples with distinct use cases (for few-shot prompts)."""
    import random
    rng = random.Random(seed)
    pool = train[:]
    rng.shuffle(pool)
    shots, seen = [], set()
    for t, l in pool:
        if l["use_case"] not in seen:
            seen.add(l["use_case"])
            shots.append((t, l))
        if len(shots) == k:
            break
    return shots


def run_extraction(model, tok, texts, shots=None, batch_size=8):
    """Returns (list of dict|None, seconds_per_example)."""
    if shots:
        prompts = [N.few_shot_prompt(t, shots) for t in texts]
    else:
        prompts = [N.extract_prompt(t) for t in texts]
    t0 = time.time()
    raw = generate(model, tok, prompts, max_new_tokens=128,
                   batch_size=batch_size, stop=["}"])
    dt = (time.time() - t0) / max(len(texts), 1)
    preds = [N.parse_model_output(r) for r in raw]
    return preds, dt, raw


# ---------------------------------------------------------- fine-tuning
class _SFTData(torch.utils.data.Dataset):
    def __init__(self, tok, pairs, max_len=640):
        self.items = []
        for prompt, completion in pairs:
            p = tok(prompt, add_special_tokens=True)["input_ids"]
            c = tok(" " + completion.strip(), add_special_tokens=False)["input_ids"]
            c = c + [tok.eos_token_id]
            if len(c) >= max_len:
                raise ValueError("Completion exceeds max_len; increase the training context length")
            p = p[-(max_len - len(c)):]
            ids = p + c
            labels = [-100] * len(p) + c    # loss only on completion
            self.items.append((ids, labels))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def _collate(pad_id):
    def fn(batch):
        m = max(len(x[0]) for x in batch)
        ids = torch.full((len(batch), m), pad_id, dtype=torch.long)
        lab = torch.full((len(batch), m), -100, dtype=torch.long)
        att = torch.zeros((len(batch), m), dtype=torch.long)
        for i, (a, b) in enumerate(batch):          # right padding for training
            ids[i, :len(a)] = torch.tensor(a)
            lab[i, :len(b)] = torch.tensor(b)
            att[i, :len(a)] = 1
        return dict(input_ids=ids, labels=lab, attention_mask=att)
    return fn


def finetune_lora(name, pairs, out_dir, epochs=2, lr=2e-4, batch_size=8,
                  grad_accum=2, r=16, max_len=640, grad_ckpt=False, seed=42):
    """LoRA fine-tune OPT `name` on (prompt, completion) pairs.
    Returns (train_seconds, final_loss)."""
    from transformers import (AutoTokenizer, AutoModelForCausalLM,
                              TrainingArguments, Trainer, set_seed)
    from peft import LoraConfig, get_peft_model
    set_seed(seed)
    if not pairs:
        raise ValueError("No training pairs supplied")
    cuda = torch.cuda.is_available()
    tok = AutoTokenizer.from_pretrained(name, use_fast=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16 if cuda else torch.float32)
    model.to("cuda" if cuda else "cpu")
    model.config.use_cache = False
    if grad_ckpt:
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
    cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, bias="none",
                     target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
                     task_type="CAUSAL_LM")
    model = get_peft_model(model, cfg)
    for p in model.parameters():                      # fp16 base, fp32 adapters
        if p.requires_grad:
            p.data = p.data.float()
    model.print_trainable_parameters()

    ds = _SFTData(tok, pairs, max_len)
    args = TrainingArguments(
        output_dir=out_dir + "_ckpt", per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum, num_train_epochs=epochs,
        learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.05 * epochs * ((len(ds) + batch_size * grad_accum - 1) // (batch_size * grad_accum)))),
        fp16=cuda, logging_steps=20, save_strategy="no", report_to="none",
        remove_unused_columns=False, optim="adamw_torch", seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=ds,
                      data_collator=_collate(tok.pad_token_id))
    t0 = time.time()
    result = trainer.train()
    secs = time.time() - t0
    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)
    loss = float(result.training_loss)
    del model, trainer
    free()
    return secs, loss


def load_finetuned(name, adapter_dir):
    """Base OPT + LoRA adapter, merged into fp16 weights for fast inference."""
    from peft import PeftModel
    tok, base = load_opt(name)
    model = PeftModel.from_pretrained(base, adapter_dir)
    model = model.merge_and_unload()
    model.eval()
    return tok, model


# -------------------------------------------------------- training pairs
def extraction_pairs(dataset):
    return [(N.extract_prompt(t), N.to_json(l)) for t, l in dataset]


def explanation_pairs(engine, dataset, pc_engine):
    """(facts -> explanation) pairs from real engine output."""
    pairs = []
    for t, l in dataset:
        res = engine.recommend(l)
        if not res["ok"]:
            continue
        pairs.append((N.explain_prompt(pc_engine.facts_text(res)),
                      pc_engine.template_explanation(res)))
    return pairs


# ----------------------------------------------------------- grounding check
_NUM = re.compile(r"\d[\d,]*(?:\.\d+)?")
_MODEL = re.compile(
    r"\b(?:RTX|GTX|RX|GT|Arc|Ryzen|Core(?: Ultra)?|Xeon|i[3579]-?|Radeon|GeForce)"
    r"[\w\- ]*?\d[\w\-]*", re.I)


def _sig_numbers(text):
    out = set()
    for m in _NUM.findall(text):
        v = m.replace(",", "").rstrip(".")
        if len(v.replace(".", "")) >= 2:        # ignore 1-digit numbers
            out.add(v)
    return out


def _models(text):
    return {re.sub(r"\s+", " ", m).strip().lower() for m in _MODEL.findall(text)}


def check_grounded(text, facts):
    """Lexical grounding: every significant number and every CPU/GPU model
    mentioned in `text` must also appear in `facts`.
    Returns (is_grounded, unsupported_numbers, unsupported_models)."""
    fnums = _sig_numbers(facts)
    ftxt = facts.lower()
    bad_n = sorted(n for n in _sig_numbers(text) if n not in fnums)
    bad_m = sorted(m for m in _models(text) if m not in ftxt)
    return (not bad_n and not bad_m), bad_n, bad_m


# ----------------------------------------------------------- inference wrapper
class OptNLU:
    """Extract requirements + explain builds with fine-tuned OPT, with
    rule-based fallbacks so the demo never returns nothing."""

    def __init__(self, extract_model, extract_tok, explain_model=None,
                 explain_tok=None):
        self.em, self.et = extract_model, extract_tok
        self.xm, self.xt = explain_model or extract_model, explain_tok or extract_tok

    def extract(self, text):
        if self.em is None:
            return N.rule_extract(text), True, "OPT not loaded"
        preds, _, raw = run_extraction(self.em, self.et, [text], batch_size=1)
        p = preds[0]
        if p is None:
            return N.rule_extract(text), True, raw[0]
        p = N.normalise(p)
        if p["budget"] is None:
            fb = N.rule_extract(text)
            p["budget"] = fb["budget"]
            p["currency"] = fb["currency"]
            return p, True, raw[0]
        return p, False, raw[0]

    def explain(self, res, pc_engine):
        if self.xm is None:
            return pc_engine.template_explanation(res), False, ([], [])
        facts = pc_engine.facts_text(res)
        txt = generate(self.xm, self.xt, [N.explain_prompt(facts)],
                       max_new_tokens=260, batch_size=1)[0].split("\n\n")[0].strip()
        ok, bn, bm = check_grounded(txt, facts)
        if not ok or len(txt) < 40:
            return pc_engine.template_explanation(res), False, (bn, bm)
        return txt, True, ([], [])

## 5. Load the supplied datasets

In [ ]:
from pathlib import Path
import zipfile
CSV_DIR = Path("csv")
required = ["cpu", "video-card", "motherboard", "memory", "internal-hard-drive", "power-supply", "case", "cpu-cooler", "monitor", "keyboard", "mouse"]
if not all((CSV_DIR / (f + ".csv")).is_file() for f in required):
    archive = Path("csv.zip")
    if not archive.is_file():
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError("Place csv.zip alongside the notebook, then rerun this cell.")
        print("Upload the supplied csv.zip")
        files.upload()
    with zipfile.ZipFile(archive) as z:
        for name in required:
            member = "csv/" + name + ".csv"
            if member not in z.namelist():
                raise ValueError("Missing dataset member: " + member)
            z.extract(member, ".")
assert all((CSV_DIR / (f + ".csv")).is_file() for f in required)
print("Required files:", len(required), "| directory:", CSV_DIR.resolve())

## 6. Data cleaning summary

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
import importlib
import pc_engine as E
importlib.reload(E)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
data = E.load_data(CSV_DIR)
rep = E.cleaning_report(CSV_DIR, data)
display(rep)
ax = rep.set_index('category')[['raw_rows','usable_rows']].plot.bar(logy=True, figsize=(9,3.5), title='Rows before / after cleaning (log scale)')
plt.tight_layout(); plt.show()
rep.set_index('category')['missing_price_pct'].plot.bar(figsize=(9,3), color='tomato', title='% of rows with no price (dropped)'); plt.tight_layout(); plt.show()

## 7. Inspect derived features

In [ ]:
# What the cleaning derives (data representation)
display(data['cpu'][['name','price','cores','boost','microarchitecture','socket','has_igpu','gaming_score','multi_score']].head(5))
display(data['mobo'][['name','price','socket','chipset_code','ddr','ddr_confirmed','form_factor','tier']].head(5))
display(data['ram'][['name','price','ddr','mhz','n_mod','gb_each','total_gb']].head(5))
display(data['gpu'][['name','chipset','vram','price','perf','tdp','brand','score']].sort_values('score', ascending=False).head(5))

## 8. Build engine demonstration

In [ ]:
engine = E.BuildEngine(data)
demo = [dict(use_case='gaming', budget=4000, currency='MYR'),
        dict(use_case='video_editing', budget=1500, currency='USD', cpu_brand='amd'),
        dict(use_case='gaming', budget=7000, currency='MYR', gpu_brand='nvidia', extras=['monitor','keyboard','mouse']),
        dict(use_case='office', budget=2500, currency='MYR')]
for r in demo:
    print('\n###', r); print(E.format_build(engine.recommend(r)))

## 9. Compatibility checks

In [ ]:
# Independent compatibility test: random requirements -> build -> re-validate every rule
import time; t=time.time()
ct = E.run_compat_tests(engine, n=300 if RUN_MODE == "full" else 30, seed=0)
print(f'{len(ct)} random requests in {time.time()-t:.0f}s')
print('built:', int(ct.built.sum()), '| builds with a compatibility violation:', int((ct.violations>0).sum()))
print('Not built (budget too low / impossible combination):')
display(ct[~ct.built][['use_case','budget','cpu_brand','gpu_brand','ram_gb','compact']])

## 10. Training and test sets

In [ ]:
import nlu_data as N, random, json
importlib.reload(N)
train_ex = N.build_dataset(2000 if RUN_MODE == "full" else 160, seed=1, mode="train")
test_syn = N.build_dataset(150 if RUN_MODE == "full" else 24, seed=99, mode="test")
train_texts = {t for t, _ in train_ex}
test_syn = [(t, l) for t, l in test_syn if t not in train_texts]
test_ood = N.OOD_SET
print(len(train_ex), len(test_syn), len(test_ood))
for t,l in train_ex[:4]: print('\n', t, '\n  ', N.to_json(l))
pd.Series([l['use_case'] for _,l in train_ex]).value_counts().plot.bar(figsize=(6,2.5), title='Training label balance (use_case)'); plt.show()

## 11. Rule-based baseline

In [ ]:
results = {}
def record(model, setting, tname, ds, preds, dt=float('nan')):
    m = N.evaluate(preds, [l for _,l in ds]); m['sec/example'] = dt
    results[(model, setting, tname)] = m
for tname, ds in [('synthetic', test_syn), ('OOD', test_ood)]:
    record('Rule-based', 'regex', tname, ds, [N.rule_extract(t) for t,_ in ds])
pd.DataFrame(results).T[['json_valid_%','exact_match_%','use_case_%','budget_%','mean_field_acc_%']].round(1)

## 12. Zero-shot and few-shot OPT evaluation

In [ ]:
import opt_nlu as O
shots = O.pick_shots(train_ex, k=4)
importlib.reload(O)
if not SIZES: print("Skipping OPT evaluation in baseline mode.")
for size in SIZES:
    tok, model = O.load_opt(O.OPT_SIZES[size])
    print(f'OPT-{size}: {O.n_params(model)/1e6:.0f}M params')
    for setting, sh in [('zero-shot', None), ('few-shot', shots)]:
        for tname, ds in [('synthetic', test_syn), ('OOD', test_ood)]:
            preds, dt, raw = O.run_extraction(model, tok, [t for t,_ in ds], shots=sh, batch_size=4)
            record(f'OPT-{size}', setting, tname, ds, preds, dt)
            if setting == 'zero-shot' and tname == 'OOD':
                print('  sample zero-shot output:', repr(raw[0][:150]))
    del model, tok; O.free()
pd.DataFrame(results).T[['json_valid_%','exact_match_%','mean_field_acc_%','sec/example']].round(1)

## 13. LoRA extraction training

In [ ]:
train_pairs = O.extraction_pairs(train_ex)
ft_info = {}
for size in SIZES:
    big = size == "1.3b"
    secs, loss = O.finetune_lora(O.OPT_SIZES[size], train_pairs, f"adapters/extract_{size}",
                                 epochs=2 if RUN_MODE == "full" else 1,
                                 batch_size=4 if big else 8, grad_accum=4 if big else 2,
                                 grad_ckpt=big)
    ft_info[size] = dict(train_seconds=round(secs), training_loss=round(loss, 4))
    print(size, ft_info[size])
display(pd.DataFrame(ft_info).T)

## 14. Evaluate fine-tuned extraction

In [ ]:
ood_raw = None
for size in SIZES:
    tok, model = O.load_finetuned(O.OPT_SIZES[size], f'adapters/extract_{size}')
    for tname, ds in [('synthetic', test_syn), ('OOD', test_ood)]:
        preds, dt, raw = O.run_extraction(model, tok, [t for t,_ in ds], batch_size=4)
        record(f'OPT-{size}', 'LoRA fine-tuned', tname, ds, preds, dt)
        if tname == 'OOD': ood_raw = (size, ds, preds)
    del model, tok; O.free()
res_df = pd.DataFrame(results).T
res_df.index.names = ['model','setting','test set']
display(res_df[['json_valid_%','exact_match_%','use_case_%','budget_%','mean_field_acc_%','sec/example']].round(1))

## 15. Compare accuracy

In [ ]:
# Comparison chart
piv = res_df['exact_match_%'].unstack('test set').round(1)
piv.plot.barh(figsize=(8, 6), title='Exact-match accuracy of requirement extraction'); plt.xlabel('% of requests where ALL fields correct'); plt.tight_layout(); plt.show()

## 16. Out-of-domain error analysis

In [ ]:
# Error analysis on the final configured fine-tuned extraction model.
if ood_raw is not None:
    size, ds, preds = ood_raw
    errors = []
    for (t, g), p in zip(ds, preds):
        pn, gn = N.normalise(p), N.normalise(g)
        bad = {k: (gn[k], pn[k]) for k in N.KEYS if pn[k] != gn[k]}
        if p is None: bad["parse"] = ("valid JSON", "invalid JSON")
        if bad: errors.append((t, bad))
    for t, bad in errors[:12]: print(t, "\n   (gold, predicted):", bad)
    print(f"Incorrect OOD requests: {len(errors)}/{len(ds)} (first 12 displayed)")
else:
    print("OPT error analysis skipped in baseline mode.")

## 17. Train the explanation adapter

In [ ]:
if EXPLAIN_SIZE:
    ex_train = N.build_dataset(500 if RUN_MODE == "full" else 40, seed=7, mode="train")
    t0 = time.time()
    ex_pairs = O.explanation_pairs(engine, ex_train, E)
    print(len(ex_pairs), "training pairs built in", round(time.time()-t0), "s")
    if not ex_pairs: raise ValueError("No successful builds for explanation training")
    print(ex_pairs[0][0]); print(ex_pairs[0][1])
    secs, loss = O.finetune_lora(O.OPT_SIZES[EXPLAIN_SIZE], ex_pairs, f"adapters/explain_{EXPLAIN_SIZE}",
                                 epochs=2 if RUN_MODE == "full" else 1,
                                 batch_size=2, grad_accum=4, grad_ckpt=True, max_len=1024)
    print("Explanation training seconds:", round(secs), "mean training loss:", round(loss, 4))
else:
    print("Explanation training skipped in baseline mode.")

## 18. Evaluate generated explanations

In [ ]:
explanation_results = pd.DataFrame()
if EXPLAIN_SIZE:
    # Evaluate: 60 unseen requests -> engine -> facts -> OPT explanation
    ex_test = [l for _,l in N.build_dataset(60 if RUN_MODE == "full" else 12, seed=123, mode='test')] + [l for _,l in test_ood[:20 if RUN_MODE == 'full' else 5]]
    builds = [engine.recommend(l) for l in ex_test]
    builds = [b for b in builds if b['ok']]
    facts = [E.facts_text(b) for b in builds]
    prompts = [N.explain_prompt(f) for f in facts]
    rows, samples = [], {}
    for label, loader in [(f'OPT-{EXPLAIN_SIZE} zero-shot', lambda: O.load_opt(O.OPT_SIZES[EXPLAIN_SIZE])),
                          (f'OPT-{EXPLAIN_SIZE} LoRA-tuned', lambda: O.load_finetuned(O.OPT_SIZES[EXPLAIN_SIZE], f'adapters/explain_{EXPLAIN_SIZE}'))]:
        tok, model = loader()
        outs = [o.split('\n\n')[0].strip() for o in O.generate(model, tok, prompts, max_new_tokens=260, batch_size=4)]
        g = [O.check_grounded(o, f) for o, f in zip(outs, facts)]
        rows.append(dict(model=label, n=len(outs),
                         lexical_check_pct=round(100*sum(x[0] and len(o) >= 40 for x,o in zip(g,outs))/len(g), 1),
                         avg_unsupported_numbers=round(sum(len(x[1]) for x in g)/len(g), 2),
                         avg_unsupported_models=round(sum(len(x[2]) for x in g)/len(g), 2),
                         avg_words=round(sum(len(o.split()) for o in outs)/len(outs), 1)))
        samples[label] = outs[0]
        del model, tok; O.free()
    explanation_results = pd.DataFrame(rows)
    display(explanation_results)
    print('FACTS:\n' + facts[0]);
    for k, v in samples.items(): print('\n[' + k + ']\n' + v)
else:
    print("OPT explanation evaluation skipped in baseline mode.")

## 19. End-to-end advisor

In [ ]:
if EXPLAIN_SIZE:
    tok_e, model_e = O.load_finetuned(O.OPT_SIZES[EXPLAIN_SIZE], f"adapters/extract_{EXPLAIN_SIZE}")
    tok_x, model_x = O.load_finetuned(O.OPT_SIZES[EXPLAIN_SIZE], f"adapters/explain_{EXPLAIN_SIZE}")
    nlu = O.OptNLU(model_e, tok_e, model_x, tok_x)
else:
    nlu = O.OptNLU(None, None)

def advise(text):
    req, fallback, raw = nlu.extract(text)
    if not req.get('budget'):
        res = dict(ok=False, message="Please specify a positive budget, e.g. RM 4000 or USD 1000.")
        return req, res, res['message'], fallback, False
    res = engine.recommend(req)
    expl, from_opt, flags = nlu.explain(res, E) if res['ok'] else (res['message'], False, None)
    return req, res, expl, fallback, from_opt

for q in ["boss rm5000 got any good gaming pc? intel cpu and 32gb ram",
          "pc untuk edit video tiktok, bajet 4 ribu",
          "I want a quiet workstation for coding, $1500, AMD CPU please"]:
    req, res, expl, fb, from_opt = advise(q)
    print('\n>>', q); print('requirement:', req, '| extraction fallback:', fb)
    print(E.format_build(res)); print('explanation (OPT-generated, grounded):' , from_opt); print(expl)

## 20. End-to-end build matching metric

In [ ]:
# End-to-end: compare a successful predicted build to the gold-input build.
def build_names(r):
    if not r.get("budget"): return None
    res = engine.recommend(r)
    return tuple(it["name"] for it in res["items"]) if res["ok"] else None
scores = {"Rule-based": 0}
if EXPLAIN_SIZE: scores["OPT + fallback"] = 0
eligible = 0
for t, gold in test_ood:
    g = build_names(gold)
    if g is None: continue
    eligible += 1
    scores["Rule-based"] += build_names(N.rule_extract(t)) == g
    if EXPLAIN_SIZE: scores["OPT + fallback"] += build_names(nlu.extract(t)[0]) == g
end_to_end = pd.DataFrame([dict(model=k, exact_build_matches=v, eligible_gold_builds=eligible,
                               exact_build_pct=100*v/eligible if eligible else float("nan")) for k,v in scores.items()])
display(end_to_end)

## 21. Optional Gradio prototype

In [ ]:
def to_md(res):
    if not res['ok']: return res['message']
    rows = ['| Part | Product | Spec | Price (RM) |', '|---|---|---|---|']
    for it in res['items']:
        rows.append(f"| {E.CAT_LABEL[it['category']]} | {it['name']} | {it['spec']} | {it['price_myr']:,.0f} |")
    rows.append(f"| **Total** | | | **{res['total_myr']:,.0f}** (budget {res['budget_myr']:,.0f}) |")
    return '\n'.join(rows + [''] + ['- ' + u for u in res['upgrades']] + ['- :warning: ' + w for w in res['warnings']])

def ui(text):
    req, res, expl, fb, from_opt = advise(text)
    tag = '\n\n_Explanation: OPT-generated; passed the limited lexical check._' if from_opt else '\n\n_Explanation: template or failure message (not OPT-generated)._' 
    tag += '\n\n_Requirement extraction: rule-based fallback used._' if fb else '\n\n_Requirement extraction: OPT._' 
    return json.dumps(req, indent=2), to_md(res), expl + tag

if LAUNCH_UI:
    import gradio as gr
    demo_ui = gr.Interface(ui, gr.Textbox(lines=3, label="Describe your PC (English / Malay / Manglish)"),
                    [gr.Code(label="Extracted requirements", language="json"), gr.Markdown(label="Recommended build"), gr.Textbox(label="Explanation")],
                    examples=[["nak build PC untuk main game, bajet RM4500, nak AMD"], ["Need a PC for video editing, budget $1800, with monitor"]],
                    title="PC Build Advisor (OPT)")
    demo_ui.launch(share=SHARE_UI, prevent_thread_lock=True, debug=False)
else:
    print("Gradio launch skipped. Set LAUNCH_UI=True and rerun this cell to launch.")

## 22. Export results and adapters

In [ ]:
# Save actual results and a self-contained source/results archive (no automatic download).
from pathlib import Path
import zipfile
out = Path("pc_advisor_results")
out.mkdir(exist_ok=True)
res_df.round(2).to_csv(out / "extraction_results.csv")
ct.to_csv(out / "compat_test.csv", index=False)
rep.to_csv(out / "cleaning_report.csv", index=False)
end_to_end.to_csv(out / "end_to_end.csv", index=False)
if not explanation_results.empty:
    explanation_results.to_csv(out / "explanation_results.csv", index=False)
else:
    (out / "explanation_results.csv").unlink(missing_ok=True)
pd.DataFrame(ft_info).T.to_csv(out / "training_results.csv")
(out / "run_mode.txt").write_text(f"Run mode: {RUN_MODE}\nModels: {SIZES}\nUSD_TO_MYR: {E.USD_TO_MYR} (fixed assumption, not live pricing)\n")
with zipfile.ZipFile("pc_advisor_results.zip", "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(out.iterdir()): z.write(path, path.name)
    for path in [Path("pc_engine.py"), Path("nlu_data.py"), Path("opt_nlu.py")]: z.write(path, path.name)
    for size in SIZES:
        for kind in (["extract", "explain"] if size == EXPLAIN_SIZE else ["extract"]):
            for path in Path(f"adapters/{kind}_{size}").rglob("*"):
                if path.is_file(): z.write(path, str(path))
print("Saved:", Path("pc_advisor_results.zip").resolve())
# In Colab, download pc_advisor_results.zip from the Files sidebar.